# Strands 에이전트와 AgentCore 메모리 (단기 메모리) - MemoryManager 사용


## 소개

이 튜토리얼은 **MemoryManager**와 **MemorySessionManager**를 사용하여 AgentCore **단기 메모리**로 Strands 에이전트를 활용한 **개인 에이전트**를 구축하는 방법을 시연합니다. 에이전트는 `get_last_k_turns`를 사용하여 세션의 최근 대화를 기억하고, 사용자가 돌아올 때 원활하게 대화를 계속할 수 있습니다.

**참고: 이 샘플은 MemoryManager 및 MemorySessionManager를 사용하는 단기 메모리 샘플 버전입니다.**


### 튜토리얼 세부 정보

| 정보                | 세부 사항                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 단기 대화형                                                                      |
| 에이전트 유형       | 개인 에이전트                                                                    |
| 에이전트 프레임워크 | Strands Agents                                                                   |
| LLM 모델            | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소  | MemoryManager를 사용한 AgentCore 단기 메모리, AgentInitializedEvent 및 MessageAddedEvent 훅 |
| 예제 복잡도         | 초급                                                                             |

학습 내용:
- MemoryManager를 사용한 대화 연속성을 위한 단기 메모리 사용
- MemorySessionManager를 사용한 최근 K개 대화 턴 검색
- 실시간 정보를 위한 웹 검색 도구
- 세션 관리를 사용한 대화 이력으로 에이전트 초기화
- MemoryClient에서 MemoryManager 아키텍처로의 마이그레이션에 활용 가능

## 아키텍처
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## 사전 요구사항

이 튜토리얼을 실행하려면 다음이 필요합니다:
- Python 3.10+
- AgentCore 메모리 권한이 있는 AWS 자격 증명
- MemoryManager 지원이 포함된 Amazon Bedrock AgentCore SDK
- Amazon Bedrock 모델 접근 권한

환경 설정을 시작합니다!

## 1단계: 설정 및 임포트

In [2]:
!pip install -qr requirements.txt

In [ ]:
import logging
from datetime import datetime
from botocore.exceptions import ClientError

# 로깅 설정
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("personal-agent")

In [ ]:
# Strands Agent에 필요한 모듈 임포트
import os
from strands import Agent, tool
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent

# 메모리 관리 모듈 임포트
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole
from bedrock_agentcore.memory.session import MemorySession, MemorySessionManager

# 메시지 역할 상수 정의
USER = MessageRole.USER
ASSISTANT = MessageRole.ASSISTANT

# 구성
REGION = os.getenv('AWS_REGION', 'us-east-1') # 에이전트의 AWS 리전
ACTOR_ID = "user_123" # 고유 식별자 (에이전트 ID, 사용자 ID 등)
SESSION_ID = "personal_session_001" # 고유 세션 식별자

# IAM 역할 생성을 위한 boto3 임포트
import boto3
import json as json_module

## 2단계: 웹 검색 도구

먼저 에이전트를 위한 간단한 웹 검색 도구를 만듭니다. 이 부분은 원래 구현에서 변경되지 않았습니다.

In [ ]:
from ddgs.exceptions import DDGSException, RatelimitException
from ddgs import DDGS

@tool
def websearch(keywords: str, region: str = "us-en", max_results: int = 5) -> str:
    """Search the web for updated information.
    
    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc..
        max_results (int | None): The maximum number of results to return.
    Returns:
        List of dictionaries with search results.
    
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except RatelimitException:
        return "Rate limit reached. Please try again later."
    except DDGSException as e:
        return f"Search error: {e}"
    except Exception as e:
        return f"Search error: {str(e)}"

logger.info("웹 검색 도구 준비 완료")

## 3단계: MemoryManager를 사용하여 메모리 리소스 생성

단기 메모리의 경우 MemoryManager를 사용하여 전략 없이 메모리 리소스를 생성합니다. 이는 `get_last_k_turns`로 검색할 수 있는 원시 대화 턴을 저장합니다.

**참고: 이 섹션은 레거시 MemoryClient 대신 MemoryManager 아키텍처를 사용합니다.**

In [ ]:
# Memory Manager 초기화
memory_manager = MemoryManager(region_name=REGION)
memory_name = "PersonalAgentMemoryManager"

logger.info(f"MemoryManager 초기화 완료 (리전: {REGION})")
logger.info(f"메모리 매니저 유형: {type(memory_manager)}")

# MemoryManager를 사용하여 메모리 리소스 생성
logger.info(f"단기 대화 저장을 위한 메모리 '{memory_name}' 생성 중...")

try:
    memory = memory_manager.get_or_create_memory(
        name=memory_name,
        strategies=[],  # 단기 메모리에는 전략 없음
        description="Short-term memory for personal agent",
        event_expiry_days=7,  # 단기 메모리 보존 기간
        memory_execution_role_arn=None,  # 단기 메모리에는 선택사항
    )
    memory_id = memory.id
    logger.info(f"MemoryManager로 메모리 생성/검색 성공:")
    logger.info(f"   메모리 ID: {memory_id}")
    logger.info(f"   메모리 이름: {memory.name}")
    logger.info(f"   메모리 상태: {memory.status}")
    
except Exception as e:
    # 향상된 오류 보고로 메모리 생성 중 오류 처리
    logger.error(f"메모리 생성 실패: {e}")
    logger.error(f"오류 유형: {type(e).__name__}")
    import traceback
    traceback.print_exc()
    
    # 오류 시 정리 - 부분적으로 생성된 메모리 삭제
    if 'memory_id' in locals():
        try:
            logger.info(f"부분적으로 생성된 메모리 정리 시도: {memory_id}")
            memory_manager.delete_memory(memory_id)
            logger.info(f"메모리 정리 성공: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"메모리 정리 실패: {cleanup_error}")
    
    # 원래 예외를 다시 발생시킴
    raise

## 4단계: 세션 매니저 초기화

이 섹션에서는 세션 기반 메모리 작업을 위한 MemorySessionManager를 소개하고 액터 및 세션을 관리하기 위한 MemorySession을 생성합니다.

In [ ]:
# 세션 메모리 매니저 초기화
session_manager = MemorySessionManager(memory_id=memory.id, region_name=REGION)

# 특정 액터/세션 조합에 대한 메모리 세션 생성
user_session = session_manager.create_memory_session(
    actor_id=ACTOR_ID, 
    session_id=SESSION_ID
)

logger.info(f"세션 매니저 초기화 완료 (메모리: {memory.id})")
logger.info(f"메모리 세션 생성 완료 (액터: {ACTOR_ID}, 세션: {SESSION_ID})")
logger.info(f"세션 매니저 유형: {type(session_manager)}")
logger.info(f"메모리 세션 유형: {type(user_session)}")

## 5단계: 메모리 훅 프로바이더

이 단계에서는 MemorySession을 사용하여 메모리 작업을 자동화하는 커스텀 `MemoryHookProvider` 클래스를 정의합니다. 훅은 에이전트의 실행 생명주기의 특정 지점에서 실행되는 특수 함수입니다. 우리가 만드는 메모리 훅은 두 가지 주요 기능을 수행합니다:
1. **최근 대화 로드**: `AgentInitializedEvent` 훅을 사용하여 에이전트 초기화 시 최근 대화 이력을 자동으로 로드합니다.
2. **마지막 메시지 저장**: 세션 매니저를 사용하여 새로운 대화 메시지를 저장합니다.

**MemoryClient 버전과의 주요 차이점:**
- MemoryClient 대신 MemorySession 사용
- 튜플 대신 ConversationalMessage 객체 사용
- create_event() 대신 add_turns() 사용
- 타입 안전성을 위해 MessageRole enum 사용

In [ ]:
class MemoryHookProvider(HookProvider):
    def __init__(self, memory_session: MemorySession):  # MemorySession을 받음
        self.memory_session = memory_session
    
    def on_agent_initialized(self, event: AgentInitializedEvent):
        """MemorySession을 사용하여 에이전트 시작 시 최근 대화 이력 로드"""
        try:
            # 사전 구성된 메모리 세션 사용 (actor_id/session_id 불필요)
            recent_turns = self.memory_session.get_last_k_turns(k=5)
            
            if recent_turns:
                # 맥락을 위해 대화 이력 포맷
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        # EventMessage 객체와 dict 형식 모두 처리
                        if hasattr(message, 'role') and hasattr(message, 'content'):
                            role = message['role']
                            content = message['content']
                        else:
                            role = message.get('role', 'unknown')
                            content = message.get('content', {}).get('text', '')
                        context_messages.append(f"{role}: {content}")
                
                context = "\n".join(context_messages)
                # 에이전트의 시스템 프롬프트에 맥락 추가
                event.agent.system_prompt += f"\n\nRecent conversation:\n{context}"
                logger.info(f"MemorySession을 사용하여 {len(recent_turns)}개의 대화 턴 로드 완료")
                
        except Exception as e:
            logger.error(f"메모리 로드 오류: {e}")

    def on_message_added(self, event: MessageAddedEvent):
        """MemorySession을 사용하여 메모리에 메시지 저장"""
        messages = event.agent.messages
        try:
            if messages and len(messages) > 0 and messages[-1]["content"][0].get("text"):
                message_text = messages[-1]["content"][0]["text"]
                message_role = MessageRole.USER if messages[-1]["role"] == "user" else MessageRole.ASSISTANT
                
                # 메모리 세션 인스턴스 사용 (actor_id/session_id 전달 불필요)
                result = self.memory_session.add_turns(
                    messages=[ConversationalMessage(message_text, message_role)]
                )
                
                event_id = result['eventId']
                logger.info(f"메시지 저장 완료 (이벤트 ID: {event_id}, 역할: {message_role.value})")
                
        except Exception as e:
            logger.error(f"메모리 저장 오류: {e}")
            import traceback
            logger.error(f"전체 트레이스백: {traceback.format_exc()}")
    
    def register_hooks(self, registry: HookRegistry):
        # 메모리 훅 등록
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        logger.info("MemorySession으로 메모리 훅 등록 완료")

## 6단계: 웹 검색 기능을 갖춘 개인 에이전트 생성

이 에이전트는 MemorySessionManager에서 생성된 MemorySession과 함께 작동하는 MemoryHookProvider를 사용합니다.

In [ ]:
def create_personal_agent():
    """MemorySession을 사용하여 메모리 및 웹 검색 기능을 갖춘 개인 에이전트 생성"""
    agent = Agent(
        name="PersonalAssistant",
        model="global.anthropic.claude-haiku-4-5-20251001-v1:0",  # 또는 선호하는 모델
        system_prompt=f"""You are a helpful personal assistant with web search capabilities.
        
        You can help with:
        - General questions and information lookup
        - Web searches for current information
        - Personal task management
        
        When you need current information, use the websearch function.
        Today's date: {datetime.today().strftime('%Y-%m-%d')}
        Be friendly and professional.""",
        hooks=[MemoryHookProvider(user_session)], 
        tools=[websearch],
    )
    return agent

# 에이전트 생성
agent = create_personal_agent()
logger.info("MemorySession 및 웹 검색 기능을 갖춘 개인 에이전트 생성 완료")

#### 축하합니다! MemoryManager 및 MemorySession을 갖춘 에이전트가 준비되었습니다
## 에이전트를 테스트해봅시다

In [ ]:
# 메모리를 사용한 대화 테스트
print("=== 첫 번째 대화 ===")
# "제 이름은 Alex이고 AI에 대해 배우는 것에 관심이 있습니다."라는 의미의 입력
print(f"사용자: My name is Alex and I'm interested in learning about AI.")
print(f"에이전트: ", end="")
agent("My name is Alex and I'm interested in learning about AI.")

In [ ]:
# "2025년 최신 AI 트렌드를 검색해줄 수 있나요?"라는 의미의 입력
print(f"사용자: Can you search for the latest AI trends in 2025?")
print(f"에이전트: ", end="")
agent("Can you search for the latest AI trends in 2025?")

In [ ]:
# "특히 머신러닝 응용에 관심이 있습니다."라는 의미의 입력
print(f"사용자: I'm particularly interested in machine learning applications.")
print(f"에이전트: ", end="")
agent("I'm particularly interested in machine learning applications.")

## MemorySessionManager로 메모리 연속성 테스트

메모리 시스템이 올바르게 작동하는지 테스트하기 위해 에이전트의 새 인스턴스를 만들고 MemorySessionManager를 사용하여 이전에 저장된 정보에 접근할 수 있는지 확인합니다:

In [ ]:
# 새 에이전트 인스턴스 생성 (사용자가 돌아온 것을 시뮬레이션)
print("=== 사용자 복귀 - 새 세션 ===")
new_agent = create_personal_agent()

# 메모리 연속성 테스트
# "제 이름이 뭐였죠?"라는 의미의 입력
print(f"사용자: What was my name again?")
print(f"에이전트: ", end="")
new_agent("What was my name again?")

# "머신러닝에 대해 더 많은 정보를 검색해줄 수 있나요?"라는 의미의 입력
print(f"사용자: Can you search for more information about machine learning?")
print(f"에이전트: ", end="")
new_agent("Can you search for more information about machine learning?")

## MemorySession을 사용하여 저장된 메모리 보기

In [ ]:
# MemorySession을 사용하여 메모리에 저장된 내용 확인
print("=== 메모리 내용 ===")
recent_turns = user_session.get_last_k_turns(k=3) 

for i, turn in enumerate(recent_turns, 1):
    print(f"턴 {i}:")
    for message in turn:
        role = message['role']
        content = message['content']['text'][:100] + "..." if len(message['content']['text']) > 100 else message['content']['text']
        print(f"  {role}: {content}")
    print()

## 요약

이 튜토리얼에서는 MemorySessionManager와 MemorySession을 사용하여 개인 에이전트를 구축하는 방법을 보여주었습니다. 학습한 내용:

- **MemorySessionManager**: 여러 세션에 걸친 메모리 작업을 위한 상위 수준 매니저
- **MemorySession**: 반복적인 파라미터 전달을 제거하는 세션별 인터페이스. MemorySession을 사용하면 모든 메서드에 actor_id/session_id를 전달할 필요가 없습니다
- **타입 안전성**: 세션은 생성 시 특정 액터/세션에 바인딩됩니다
- **더 나은 캡슐화**: 세션별 작업이 세션 객체 내에 포함됩니다
- **메모리 훅**: 에이전트 훅은 세션 기반 아키텍처와 함께 작동할 수 있습니다
- **대화 연속성**: MemoryManager 및 MemorySession으로 단기 메모리 기능 유지

### MemorySession의 주요 이점:
1. **간소화된 API**: 모든 메서드 호출에 actor_id/session_id를 전달할 필요 없음
2. **사전 구성된 맥락**: 세션이 생성 시 특정 액터/세션에 바인딩됨
3. **일관된 인터페이스**: 모든 세션 작업이 동일한 사전 구성된 맥락 사용

## 정리 (선택사항)

In [ ]:
# MemoryManager를 사용하여 메모리 리소스를 삭제하려면 주석 해제
# try:
#     memory_manager.delete_memory(memory_id)
#     logger.info(f"메모리 삭제 완료: {memory_id}")
# except Exception as e:
#     logger.error(f"메모리 삭제 실패: {e}")